In [1]:
# ==============================================================================
# 1. INSTALACIÓN DE MÓDULOS (Solvers MILP: CBC y HiGHS)
# ==============================================================================
%pip install -q amplpy ipywidgets
!python -m amplpy.modules install coin highs

import numpy as np
import pandas as pd
from amplpy import AMPL, modules
from IPython.display import display, HTML
import ipywidgets as widgets

# ==============================================================================
# 2. GENERACIÓN DE DATOS (ESTOCÁSTICOS) - Código Original Mantenido
# ==============================================================================
np.random.seed(4186)

n_productos = 4; n_trabajadores = 2; n_meses = 12

# Demanda
media_demanda = 1650; desv_demanda = 450
demanda_base = np.random.normal(media_demanda, desv_demanda, (n_productos, n_meses))
tendencia_demanda = np.linspace(0, 0.1, n_meses)
d_it = np.maximum(0, demanda_base * (1 + tendencia_demanda)).round().astype(int)

# Tasas de Producción
medias_rp = np.array([0.40, 0.32]); desv_rp = np.array([0.04, 0.02])
rp_data = [[np.random.normal(medias_rp[j], desv_rp[j]) for j in range(n_trabajadores)] for i in range(n_productos)]
rp_final = np.maximum(0.01, np.array(rp_data)).round(2)

def gen_costos(media, desv, inc, n):
    return np.maximum(0, media + np.linspace(0, media*inc, n) + np.random.normal(0, desv, n)).round().astype(int)

costos_contratacion = np.array([gen_costos(1020, 50, 0.3, n_meses), gen_costos(1600, 80, 0.3, n_meses)])
costos_despido = np.array([gen_costos(1480, 100, 0.3, n_meses), gen_costos(2000, 120, 0.3, n_meses)])
costos_remuneracion = np.array([gen_costos(7.0, 0.2, 0.1, n_meses), gen_costos(10.0, 0.25, 0.1, n_meses)])

# Parámetros Fijos
Ingresos = {0: 14000, 1: 19000, 2: 14500, 3: 16500}
Utilizacion_Max = {0: 0.80, 1: 0.90}
W_Inicial = {0: 3, 1: 0}
W_Min = {0: 5, 1: 4}; W_Max = {0: 80, 1: 70}
Costo_Almacen_Prod = {0: 25, 1: 27, 2: 23, 3: 24}
Inv_Inicial_Prod = {0: 120, 1: 160, 2: 130, 3: 80}
Inv_Seguridad_Prod = {0: 50, 1: 50, 2: 50, 3: 50}

BOM_Data = [
    [5.0, 3.0, 5.0, 5.0], [2, 3, 4, 2.0], [2, 4.0, 6.0, 5.0],
    [4, 5, 2.0, 3.0], [6, 6, 4, 4], [3, 6, 2, 4]
]

Costo_Almacen_Ins = {0: 25, 1: 27, 2: 23, 3: 24, 4: 24, 5: 24}
Inv_Inicial_Ins = {0: 120, 1: 160, 2: 130, 3: 80, 4: 80, 5: 80}
Costo_Fijo_Prov = {0: 18000, 1: 0, 2: 0} 

Costo_Unit_Ins_Data = [
    [480, 820, 740], [500, 760, 790], [510, 830, 770],
    [490, 710, 750], [470, 800, 720], [470, 800, 720]
]

# ==============================================================================
# 3. CONSTRUCCIÓN DEL MODELO AMPL
# ==============================================================================
modules.load()
ampl = AMPL()
ampl.reset()
pd.options.display.float_format = '{:,.2f}'.format

# Definición del modelo
ampl.eval(r"""
# --- CONJUNTOS ---
set I; # Productos
set J; # Trabajadores
set T; # Meses
set K; # Insumos
set P; # Proveedores

# --- PARÁMETROS ESTOCÁSTICOS ---
param d{I, T};
param rp{I, J};
param C_contr{J, T};
param C_desp{J, T};
param C_rem{J, T};

# --- PARÁMETROS FIJOS ---
param Ingresos{I};
param U_max{J};
param W_ini{J};
param W_min{J};
param W_max{J};
param HR_Mensual := 160;
param HE_Max_Mensual := 12;
param Factor_HE := 1.45;

param C_alm_P{I};
param Inv_ini_P{I};
param Inv_seg_P{I};
param Inv_Max_Prod := 400;

param BOM{K, I};
param C_alm_I{K};
param Inv_ini_I{K};
param Inv_seg_I := 50;
param Inv_Max_Ins_Total := 10000;
param Adq_Max_Total := 300000;

param C_fijo_P{P};
param Capacidad_Prov := 120000;
param CU{K, P};

param Alpha := 0.80;
param Factor_CVP := 1.5;

# --- VARIABLES ---
var X{I, J, T} >= 0;                # Producción
var V{I, T} >= 0;                   # Ventas
var S_vp{I, T} >= 0;                # Ventas Perdidas
var InvProd{I, T} >= 0;             # Inventario Productos
var W{J, T} >= 0, integer;          # Plantilla
var C{J, T} >= 0, integer;          # Contratados
var D_var{J, T} >= 0, integer;      # Despedidos
var H_Reg{J, T} >= 0;               # Horas Regulares
var H_Ext{J, T} >= 0;               # Horas Extras
var Q{K, P, T} >= 0;                # Compras insumos
var InvIns{K, T} >= 0;              # Inventario Insumos
var Y{P, T} binary;                 # Proveedor Seleccionado

# --- COSTOS CALCULADOS AUTOMÁTICAMENTE ---
var Ingresos_Totales = sum{i in I, t in T} Ingresos[i] * V[i,t];
var Costo_CVP = sum{i in I, t in T} (Ingresos[i] * Factor_CVP) * S_vp[i,t];
var Costo_Laboral = sum{j in J, t in T} (
    C_contr[j,t] * C[j,t] + 
    C_desp[j,t] * D_var[j,t] + 
    C_rem[j,t] * H_Reg[j,t] + 
    (C_rem[j,t] * Factor_HE) * H_Ext[j,t]
);
var Costo_Almacen = sum{i in I, t in T} C_alm_P[i] * InvProd[i,t] + 
                    sum{k in K, t in T} C_alm_I[k] * InvIns[k,t];
var Costo_Adquisicion = sum{p in P, t in T} C_fijo_P[p] * Y[p,t] + 
                        sum{k in K, p in P, t in T} CU[k,p] * Q[k,p,t];

var Beneficio = Ingresos_Totales - (Costo_CVP + Costo_Laboral + Costo_Almacen + Costo_Adquisicion);

# --- FUNCIÓN OBJETIVO ---
maximize Z: Beneficio;

# --- RESTRICCIONES ---
# 1. Balance de Inventario Productos
subject to R_BalProd {i in I, t in T}:
    InvProd[i,t] == (if t == 0 then Inv_ini_P[i] else InvProd[i,t-1]) + sum{j in J} X[i,j,t] - V[i,t];

# 2 & 3. Satisfacción y Nivel de Servicio
subject to R_Demanda {i in I, t in T}:
    V[i,t] + S_vp[i,t] == d[i,t];
subject to R_Servicio {i in I, t in T}:
    S_vp[i,t] <= (1 - Alpha) * d[i,t];

# 4. Límites de Inventario Productos
subject to R_InvMinProd {i in I, t in T}: InvProd[i,t] >= Inv_seg_P[i];
subject to R_InvMaxProd {i in I, t in T}: InvProd[i,t] <= Inv_Max_Prod;

# 5. Balance Trabajadores
subject to R_BalW {j in J, t in T}:
    W[j,t] == (if t == 0 then W_ini[j] else W[j,t-1]) + C[j,t] - D_var[j,t];

# 6. Límites Plantilla
subject to R_LimWMin {j in J, t in T}: W[j,t] >= W_min[j];
subject to R_LimWMax {j in J, t in T}: W[j,t] <= W_max[j];

# 7. Capacidad Horas Requeridas vs Disponibles
subject to R_Capacidad {j in J, t in T}:
    sum{i in I} (X[i,j,t] / rp[i,j]) <= H_Reg[j,t] + H_Ext[j,t];

# 8 & 9. Disponibilidad Horas Regulares y Extras
subject to R_DispHReg {j in J, t in T}: H_Reg[j,t] <= U_max[j] * HR_Mensual * W[j,t];
subject to R_DispHExt {j in J, t in T}: H_Ext[j,t] <= HE_Max_Mensual * W[j,t];

# 10. Balance Inventario Insumos
subject to R_BalIns {k in K, t in T}:
    InvIns[k,t] == (if t == 0 then Inv_ini_I[k] else InvIns[k,t-1]) + 
    sum{p in P} Q[k,p,t] - sum{i in I} BOM[k,i] * sum{j in J} X[i,j,t];

# 11, 12, 13, 14. Limites de Insumos y Proveedores
subject to R_InvSegIns {k in K, t in T}: InvIns[k,t] >= Inv_seg_I;
subject to R_CapBodegaIns {t in T}: sum{k in K} InvIns[k,t] <= Inv_Max_Ins_Total;
subject to R_LimAdq {t in T}: sum{k in K, p in P} Q[k,p,t] <= Adq_Max_Total;
subject to R_LogicaProv {p in P, t in T}: sum{k in K} Q[k,p,t] <= Capacidad_Prov * Y[p,t];
""")

# ==============================================================================
# 4. TRASPASO DE DATOS A AMPL
# ==============================================================================
# Set Lists
ampl.set['I'] = list(range(n_productos))
ampl.set['J'] = list(range(n_trabajadores))
ampl.set['T'] = list(range(n_meses))
ampl.set['K'] = list(range(6))
ampl.set['P'] = list(range(3))

# Diccionarios Simples
ampl.param['Ingresos'] = Ingresos
ampl.param['U_max'] = Utilizacion_Max
ampl.param['W_ini'] = W_Inicial
ampl.param['W_min'] = W_Min
ampl.param['W_max'] = W_Max
ampl.param['C_alm_P'] = Costo_Almacen_Prod
ampl.param['Inv_ini_P'] = Inv_Inicial_Prod
ampl.param['Inv_seg_P'] = Inv_Seguridad_Prod
ampl.param['C_alm_I'] = Costo_Almacen_Ins
ampl.param['Inv_ini_I'] = Inv_Inicial_Ins
ampl.param['C_fijo_P'] = Costo_Fijo_Prov

# Diccionarios de Matrices Estocásticas
ampl.param['d'] = {(i, t): d_it[i, t] for i in range(n_productos) for t in range(n_meses)}
ampl.param['rp'] = {(i, j): rp_final[i, j] for i in range(n_productos) for j in range(n_trabajadores)}
ampl.param['C_contr'] = {(j, t): costos_contratacion[j, t] for j in range(n_trabajadores) for t in range(n_meses)}
ampl.param['C_desp'] = {(j, t): costos_despido[j, t] for j in range(n_trabajadores) for t in range(n_meses)}
ampl.param['C_rem'] = {(j, t): costos_remuneracion[j, t] for j in range(n_trabajadores) for t in range(n_meses)}
ampl.param['BOM'] = {(k, i): BOM_Data[k][i] for k in range(6) for i in range(n_productos)}
ampl.param['CU'] = {(k, p): Costo_Unit_Ins_Data[k][p] for k in range(6) for p in range(3)}

# ==============================================================================
# 5. RESOLUCIÓN
# ==============================================================================
try:
    ampl.set_option('solver', 'highs') # HiGHS suele ser más rápido para MILP abierto
except:
    ampl.set_option('solver', 'cbc')

print("\n--- Ejecutando Optimización ---")
ampl.solve()

# ==============================================================================
# 6. REPORTE VISUAL INTERACTIVO
# ==============================================================================
# Obtención de Métricas Financieras Pre-calculadas en AMPL
z_obj = ampl.get_objective('Z').value()
ingresos_totales = ampl.get_value('Ingresos_Totales')
c_laboral = ampl.get_value('Costo_Laboral')
c_adquisicion = ampl.get_value('Costo_Adquisicion')
c_almacen = ampl.get_value('Costo_Almacen')
c_cvp = ampl.get_value('Costo_CVP')
costo_operativo = c_laboral + c_adquisicion + c_almacen + c_cvp

# Encabezado HTML 
header_html = f"""
<div style="background-color: #1B4F72; padding: 25px; border-radius: 12px; color: white; font-family: sans-serif;">
    <h1 style="margin: 0;">Planificación Agregada: Lácteos Andinos</h1>
    <hr style="border: 0.5px solid #5DADE2;">
    <p style="font-size: 20px;">Utilidad Neta (Z): <span style="color: #F7DC6F;"><b>$ {z_obj:,.0f}</b></span></p>
    <p style="font-size: 16px;">Margen Operativo: <span style="color: #82E0AA;"><b>{(z_obj/ingresos_totales)*100:.1f}%</b></span></p>
</div>
"""

# Extracción de Planes a DataFrames
v_X = ampl.get_variable('X').get_values().to_dict()
v_V = ampl.get_variable('V').get_values().to_dict()
v_S = ampl.get_variable('S_vp').get_values().to_dict()
v_InvP = ampl.get_variable('InvProd').get_values().to_dict()

prod_data = []
for t in range(n_meses):
    fila = {'Mes': t+1}
    for i in range(n_productos):
        fila[f'Prod_P{i}'] = sum(v_X[i, j, t] for j in range(n_trabajadores))
        fila[f'Inv_P{i}'] = v_InvP[i, t]
        fila[f'Venta_P{i}'] = v_V[i, t]
        fila[f'Perdida_P{i}'] = v_S[i, t]
    prod_data.append(fila)

v_W = ampl.get_variable('W').get_values().to_dict()
v_C = ampl.get_variable('C').get_values().to_dict()
v_D = ampl.get_variable('D_var').get_values().to_dict()
v_HExt = ampl.get_variable('H_Ext').get_values().to_dict()

rrhh_data = []
for t in range(n_meses):
    fila = {'Mes': t+1}
    for j in range(n_trabajadores):
        fila[f'W_T{j}'] = v_W[j, t]
        fila[f'C_T{j}'] = v_C[j, t]
        fila[f'D_T{j}'] = v_D[j, t]
        fila[f'HE_T{j}'] = v_HExt[j, t]
    rrhh_data.append(fila)

df_desglose = pd.DataFrame([
    ("Ingresos Totales", ingresos_totales, 0.0),
    ("Costos Laborales", c_laboral, c_laboral/costo_operativo),
    ("Costos Adquisición Insumos", c_adquisicion, c_adquisicion/costo_operativo),
    ("Costos Almacenamiento", c_almacen, c_almacen/costo_operativo),
    ("Costos Ventas Perdidas", c_cvp, c_cvp/costo_operativo),
    ("COSTO TOTAL OPERATIVO", costo_operativo, 1.0)
], columns=['Ítem', 'Monto [$]', '% del Costo Total']).set_index('Ítem')

# Creación de Pestañas Visuales
dataframes = {
    'Plan de RRHH': pd.DataFrame(rrhh_data).set_index('Mes'),
    'Plan de Producción (Productos)': pd.DataFrame(prod_data).set_index('Mes'),
    'Desglose Financiero (Tabla 1.d)': df_desglose
}

def style_df(val):
    return 'color: #D3D3D3' if abs(val) < 1e-6 else 'color: black'

tabs = []
for nombre, df in dataframes.items():
    if nombre == 'Desglose Financiero (Tabla 1.d)':
        styler = df.style.format({'Monto [$]': '{:,.0f}', '% del Costo Total': '{:.2%}'}).bar(subset=['Monto [$]'], color='#F5B041')
    else:
        styler = df.style.applymap(style_df).format("{:,.1f}")
        
    out = widgets.Output()
    with out:
        display(HTML(f"<h3>{nombre}</h3>"))
        display(styler)
    tabs.append(out)

tab_control = widgets.Tab(children=tabs)
for i, nombre in enumerate(dataframes.keys()):
    tab_control.set_title(i, nombre)

display(HTML(header_html))
display(tab_control)

$ /usr/bin/python3 -m pip install -i https://pypi.ampl.com ampl_module_base ampl_module_coin ampl_module_highs
Looking in indexes: https://pypi.ampl.com
Imported ampl_module_base.
Imported ampl_module_base.
Imported ampl_module_coin.
Imported ampl_module_highs.

--- Ejecutando Optimización ---
HiGHS 1.11.0HiGHS 1.11.0: optimal solution; objective 252735737.4
274 simplex iterations
1 branching nodes
absmipgap=9.4284, relmipgap=3.73054e-08


/tmp/ipykernel_12604/671848649.py:310: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  styler = df.style.applymap(style_df).format("{:,.1f}")


In [1]:
# ==============================================================================
# 1. INSTALACIÓN DE MÓDULOS (Solvers MILP: CBC y HiGHS)
# ==============================================================================
%pip install -q amplpy ipywidgets
!python -m amplpy.modules install coin highs

import numpy as np
import pandas as pd
from amplpy import AMPL, modules
from IPython.display import display, HTML
import ipywidgets as widgets

# ==============================================================================
# 2. GENERACIÓN DE DATOS Y PARAMETRIZACIÓN (SITUACIÓN A)
# ==============================================================================
np.random.seed(4186)

n_productos = 4; n_trabajadores = 2; n_meses = 12

# Demanda
media_demanda = 1650; desv_demanda = 450
demanda_base = np.random.normal(media_demanda, desv_demanda, (n_productos, n_meses))
tendencia_demanda = np.linspace(0, 0.1, n_meses)
d_it = np.maximum(0, demanda_base * (1 + tendencia_demanda)).round().astype(int)

# Tasas de Producción
medias_rp = np.array([0.40, 0.32]); desv_rp = np.array([0.04, 0.02])
rp_data = [[np.random.normal(medias_rp[j], desv_rp[j]) for j in range(n_trabajadores)] for i in range(n_productos)]
rp_final = np.maximum(0.01, np.array(rp_data)).round(2)

def gen_costos(media, desv, inc, n):
    return np.maximum(0, media + np.linspace(0, media*inc, n) + np.random.normal(0, desv, n)).round().astype(int)

costos_contratacion = np.array([gen_costos(1020, 50, 0.3, n_meses), gen_costos(1600, 80, 0.3, n_meses)])
costos_despido = np.array([gen_costos(1480, 100, 0.3, n_meses), gen_costos(2000, 120, 0.3, n_meses)])
costos_remuneracion = np.array([gen_costos(7.0, 0.2, 0.1, n_meses), gen_costos(10.0, 0.25, 0.1, n_meses)])

# Parámetros Fijos
Ingresos = {0: 14000, 1: 19000, 2: 14500, 3: 16500}
Utilizacion_Max = {0: 0.80, 1: 0.90}
W_Inicial = {0: 3, 1: 0}
W_Min = {0: 5, 1: 4}; W_Max = {0: 80, 1: 70}
HR_Mensual = 160; HE_Max_Mensual = 12; Factor_HE = 1.45
Costo_Almacen_Prod = {0: 25, 1: 27, 2: 23, 3: 24}
Inv_Inicial_Prod = {0: 120, 1: 160, 2: 130, 3: 80}
Inv_Seguridad_Prod = {0: 50, 1: 50, 2: 50, 3: 50}
Inv_Max_Prod = 400

BOM_Data = [
    [5.0, 3.0, 5.0, 5.0], [2, 3, 4, 2.0], [2, 4.0, 6.0, 5.0],
    [4, 5, 2.0, 3.0], [6, 6, 4, 4], [3, 6, 2, 4]
]

Costo_Almacen_Ins = {0: 25, 1: 27, 2: 23, 3: 24, 4: 24, 5: 24}
Inv_Inicial_Ins = {0: 120, 1: 160, 2: 130, 3: 80, 4: 80, 5: 80}
Inv_Max_Ins_Total = 10000; Adq_Max_Total = 300000

Costo_Fijo_Prov = {0: 18000, 1: 0, 2: 0}; Capacidad_Prov = 120000
Costo_Unit_Ins_Data = [
    [480, 820, 740], [500, 760, 790], [510, 830, 770],
    [490, 710, 750], [470, 800, 720], [470, 800, 720]
]

Alpha = 0.80; Factor_CVP = 1.5

# NUEVOS PARÁMETROS SITUACIÓN A
Costo_Fijo_Ext = {0: 150000, 1: 90000, 2: 0} 
Costo_Unit_Ext_Data = [
    [30000, 29100, 29700], # Yogurth
    [18000, 17700, 18300], # Leche
    [75000, 73500, 75600], # Mantequilla
    [99000, 97500, 100500] # Queso
]

# ==============================================================================
# 3. CONSTRUCCIÓN DEL MODELO AMPL (SITUACIÓN A)
# ==============================================================================
modules.load()
ampl = AMPL()
ampl.reset()
pd.options.display.float_format = '{:,.2f}'.format

ampl.eval(r"""
# --- CONJUNTOS ---
set I; # Productos
set J; # Trabajadores
set T; # Meses
set K; # Insumos
set P; # Proveedores
set E; # Proveedores Externos (SITUACIÓN A)

# --- PARÁMETROS ESTOCÁSTICOS ---
param d{I, T};
param rp{I, J};
param C_contr{J, T};
param C_desp{J, T};
param C_rem{J, T};

# --- PARÁMETROS FIJOS ---
param Ingresos{I};
param U_max{J};
param W_ini{J};
param W_min{J};
param W_max{J};
param HR_Mensual;
param HE_Max_Mensual;
param Factor_HE;
param C_alm_P{I};
param Inv_ini_P{I};
param Inv_seg_P{I};
param Inv_Max_Prod;
param BOM{K, I};
param C_alm_I{K};
param Inv_ini_I{K};
param Inv_seg_I := 50;
param Inv_Max_Ins_Total;
param Adq_Max_Total;
param C_fijo_P{P};
param Capacidad_Prov;
param CU{K, P};
param Alpha;
param Factor_CVP;

# --- PARÁMETROS SITUACIÓN A ---
param C_fijo_E{E};
param CUE{I, E};
param BigM := 100000;

# --- VARIABLES ---
var X{I, J, T} >= 0;                # Producción Interna
var V{I, T} >= 0;                   # Ventas
var S_vp{I, T} >= 0;                # Ventas Perdidas
var InvProd{I, T} >= 0;             # Inventario Productos
var W{J, T} >= 0, integer;          # Plantilla
var C{J, T} >= 0, integer;          # Contratados
var D_var{J, T} >= 0, integer;      # Despedidos
var H_Reg{J, T} >= 0;               # Horas Regulares
var H_Ext{J, T} >= 0;               # Horas Extras
var Q{K, P, T} >= 0;                # Compras insumos MP
var InvIns{K, T} >= 0;              # Inventario Insumos
var Y{P, T} binary;                 # Proveedor MP Seleccionado

# --- NUEVAS VARIABLES SITUACIÓN A ---
var CP{I, E, T} >= 0;               # Compra de Productos a Externos
var YE{E, T} binary;                # Activación Proveedor Externo

# --- COSTOS CALCULADOS ---
var Ingresos_Totales = sum{i in I, t in T} Ingresos[i] * V[i,t];
var Costo_CVP = sum{i in I, t in T} (Ingresos[i] * Factor_CVP) * S_vp[i,t];
var Costo_Laboral = sum{j in J, t in T} (
    C_contr[j,t] * C[j,t] + 
    C_desp[j,t] * D_var[j,t] + 
    C_rem[j,t] * H_Reg[j,t] + 
    (C_rem[j,t] * Factor_HE) * H_Ext[j,t]
);
var Costo_Almacen = sum{i in I, t in T} C_alm_P[i] * InvProd[i,t] + sum{k in K, t in T} C_alm_I[k] * InvIns[k,t];
var Costo_Adquisicion = sum{p in P, t in T} C_fijo_P[p] * Y[p,t] + sum{k in K, p in P, t in T} CU[k,p] * Q[k,p,t];

# NUEVO COSTO
var Costo_Compra_Ext = sum{e in E, t in T} C_fijo_E[e] * YE[e,t] + sum{i in I, e in E, t in T} CUE[i,e] * CP[i,e,t];

var Beneficio = Ingresos_Totales - (Costo_CVP + Costo_Laboral + Costo_Almacen + Costo_Adquisicion + Costo_Compra_Ext);

# --- FUNCIÓN OBJETIVO ---
maximize Z: Beneficio;

# --- RESTRICCIONES ---
# 1. Balance de Inventario Productos (MODIFICADO SITUACIÓN A)
subject to R_BalProd {i in I, t in T}:
    InvProd[i,t] == (if t == 0 then Inv_ini_P[i] else InvProd[i,t-1]) + 
                    sum{j in J} X[i,j,t] + sum{e in E} CP[i,e,t] - V[i,t];

# Lógica Proveedor Externo (NUEVA RESTRICCIÓN SITUACIÓN A)
subject to R_LogicaExt {e in E, t in T}:
    sum{i in I} CP[i,e,t] <= BigM * YE[e,t];

# Restricciones Originales
subject to R_Demanda {i in I, t in T}: V[i,t] + S_vp[i,t] == d[i,t];
subject to R_Servicio {i in I, t in T}: S_vp[i,t] <= (1 - Alpha) * d[i,t];
subject to R_InvMinProd {i in I, t in T}: InvProd[i,t] >= Inv_seg_P[i];
subject to R_InvMaxProd {i in I, t in T}: InvProd[i,t] <= Inv_Max_Prod;
subject to R_BalW {j in J, t in T}: W[j,t] == (if t == 0 then W_ini[j] else W[j,t-1]) + C[j,t] - D_var[j,t];
subject to R_LimWMin {j in J, t in T}: W[j,t] >= W_min[j];
subject to R_LimWMax {j in J, t in T}: W[j,t] <= W_max[j];
subject to R_Capacidad {j in J, t in T}: sum{i in I} (X[i,j,t] / rp[i,j]) <= H_Reg[j,t] + H_Ext[j,t];
subject to R_DispHReg {j in J, t in T}: H_Reg[j,t] <= U_max[j] * HR_Mensual * W[j,t];
subject to R_DispHExt {j in J, t in T}: H_Ext[j,t] <= HE_Max_Mensual * W[j,t];
subject to R_BalIns {k in K, t in T}: InvIns[k,t] == (if t == 0 then Inv_ini_I[k] else InvIns[k,t-1]) + sum{p in P} Q[k,p,t] - sum{i in I} BOM[k,i] * sum{j in J} X[i,j,t];
subject to R_InvSegIns {k in K, t in T}: InvIns[k,t] >= Inv_seg_I;
subject to R_CapBodegaIns {t in T}: sum{k in K} InvIns[k,t] <= Inv_Max_Ins_Total;
subject to R_LimAdq {t in T}: sum{k in K, p in P} Q[k,p,t] <= Adq_Max_Total;
subject to R_LogicaProv {p in P, t in T}: sum{k in K} Q[k,p,t] <= Capacidad_Prov * Y[p,t];
""")

# ==============================================================================
# 4. TRASPASO DE DATOS A AMPL
# ==============================================================================
ampl.set['I'] = list(range(n_productos)); ampl.set['J'] = list(range(n_trabajadores)); ampl.set['T'] = list(range(n_meses))
ampl.set['K'] = list(range(6)); ampl.set['P'] = list(range(3)); ampl.set['E'] = list(range(3))

ampl.param['Ingresos'] = Ingresos; ampl.param['U_max'] = Utilizacion_Max; ampl.param['W_ini'] = W_Inicial
ampl.param['W_min'] = W_Min; ampl.param['W_max'] = W_Max; ampl.param['C_alm_P'] = Costo_Almacen_Prod
ampl.param['Inv_ini_P'] = Inv_Inicial_Prod; ampl.param['Inv_seg_P'] = Inv_Seguridad_Prod
ampl.param['C_alm_I'] = Costo_Almacen_Ins; ampl.param['Inv_ini_I'] = Inv_Inicial_Ins
ampl.param['C_fijo_P'] = Costo_Fijo_Prov; ampl.param['C_fijo_E'] = Costo_Fijo_Ext

ampl.param['HR_Mensual'] = HR_Mensual; ampl.param['HE_Max_Mensual'] = HE_Max_Mensual; ampl.param['Factor_HE'] = Factor_HE
ampl.param['Inv_Max_Prod'] = Inv_Max_Prod; ampl.param['Inv_Max_Ins_Total'] = Inv_Max_Ins_Total
ampl.param['Adq_Max_Total'] = Adq_Max_Total; ampl.param['Capacidad_Prov'] = Capacidad_Prov
ampl.param['Alpha'] = Alpha; ampl.param['Factor_CVP'] = Factor_CVP

ampl.param['d'] = {(i, t): d_it[i, t] for i in range(n_productos) for t in range(n_meses)}
ampl.param['rp'] = {(i, j): rp_final[i, j] for i in range(n_productos) for j in range(n_trabajadores)}
ampl.param['C_contr'] = {(j, t): costos_contratacion[j, t] for j in range(n_trabajadores) for t in range(n_meses)}
ampl.param['C_desp'] = {(j, t): costos_despido[j, t] for j in range(n_trabajadores) for t in range(n_meses)}
ampl.param['C_rem'] = {(j, t): costos_remuneracion[j, t] for j in range(n_trabajadores) for t in range(n_meses)}
ampl.param['BOM'] = {(k, i): BOM_Data[k][i] for k in range(6) for i in range(n_productos)}
ampl.param['CU'] = {(k, p): Costo_Unit_Ins_Data[k][p] for k in range(6) for p in range(3)}
ampl.param['CUE'] = {(i, e): Costo_Unit_Ext_Data[i][e] for i in range(n_productos) for e in range(3)}

# ==============================================================================
# 5. RESOLUCIÓN
# ==============================================================================
try:
    ampl.set_option('solver', 'highs') 
except:
    ampl.set_option('solver', 'cbc')

print("\n--- Ejecutando Optimización (Situación A) ---")
ampl.solve()

# ==============================================================================
# 6. REPORTE VISUAL INTERACTIVO
# ==============================================================================
z_obj = ampl.get_objective('Z').value()
ingresos_totales = ampl.get_value('Ingresos_Totales')
c_laboral = ampl.get_value('Costo_Laboral')
c_adquisicion = ampl.get_value('Costo_Adquisicion')
c_almacen = ampl.get_value('Costo_Almacen')
c_cvp = ampl.get_value('Costo_CVP')
c_comp_ext = ampl.get_value('Costo_Compra_Ext')
costo_operativo = c_laboral + c_adquisicion + c_almacen + c_cvp + c_comp_ext

header_html = f"""
<div style="background-color: #1B4F72; padding: 25px; border-radius: 12px; color: white; font-family: sans-serif;">
    <h1 style="margin: 0;">Planificación Agregada: Situación A (Proveedores Externos)</h1>
    <hr style="border: 0.5px solid #5DADE2;">
    <p style="font-size: 20px;">Utilidad Neta (Z): <span style="color: #F7DC6F;"><b>$ {z_obj:,.0f}</b></span></p>
    <p style="font-size: 16px;">Margen Operativo: <span style="color: #82E0AA;"><b>{(z_obj/ingresos_totales)*100:.1f}%</b></span></p>
</div>
"""

# Extracción de Compras a Proveedores Externos
v_YE = ampl.get_variable('YE').get_values().to_dict()
v_CP = ampl.get_variable('CP').get_values().to_dict()

compras_ext_data = []
for t in range(n_meses):
    fila = {'Mes': t+1}
    for e in range(3):
        fila[f'Uso_Ext_{e}'] = v_YE[e, t]
    for i in range(n_productos):
        fila[f'Comprado_Prod_{i}'] = sum(v_CP[i, e, t] for e in range(3))
    compras_ext_data.append(fila)

df_desglose = pd.DataFrame([
    ("Ingresos Totales", ingresos_totales, 0.0),
    ("Costos Laborales", c_laboral, c_laboral/costo_operativo),
    ("Costos Adquisición Insumos (MP)", c_adquisicion, c_adquisicion/costo_operativo),
    ("Costos Compra Productos Externos", c_comp_ext, c_comp_ext/costo_operativo),
    ("Costos Almacenamiento", c_almacen, c_almacen/costo_operativo),
    ("Costos Ventas Perdidas", c_cvp, c_cvp/costo_operativo),
    ("COSTO TOTAL OPERATIVO", costo_operativo, 1.0)
], columns=['Ítem', 'Monto [$]', '% del Costo Total']).set_index('Ítem')

# Creación de Pestañas Visuales
dataframes = {
    'Desglose Financiero (Situación A)': df_desglose,
    'Compras a Externos (Situación A)': pd.DataFrame(compras_ext_data).set_index('Mes')
}

def style_df(val):
    return 'color: #D3D3D3' if abs(val) < 1e-6 else 'color: black'

tabs = []
for nombre, df in dataframes.items():
    if nombre == 'Desglose Financiero (Situación A)':
        styler = df.style.format({'Monto [$]': '{:,.0f}', '% del Costo Total': '{:.2%}'}).bar(subset=['Monto [$]'], color='#F5B041')
    else:
        styler = df.style.applymap(style_df).format("{:,.0f}")
        
    out = widgets.Output()
    with out:
        display(HTML(f"<h3>{nombre}</h3>"))
        display(styler)
    tabs.append(out)

tab_control = widgets.Tab(children=tabs)
for i, nombre in enumerate(dataframes.keys()):
    tab_control.set_title(i, nombre)

display(HTML(header_html))
display(tab_control)

$ /usr/bin/python3 -m pip install -i https://pypi.ampl.com ampl_module_base ampl_module_coin ampl_module_highs
Looking in indexes: https://pypi.ampl.com
Imported ampl_module_base.
Imported ampl_module_base.
Imported ampl_module_coin.
Imported ampl_module_highs.

--- Ejecutando Optimización (Situación A) ---
HiGHS 1.11.0HiGHS 1.11.0: optimal solution; objective 280652656.5
2689 simplex iterations
7 branching nodes
absmipgap=26705.9, relmipgap=9.51564e-05


/tmp/ipykernel_14630/1553878228.py:290: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  styler = df.style.applymap(style_df).format("{:,.0f}")
